# Week 2, Day 1: One Tool, Three Implementations

## Today

By the end of this session you can:

- Run the Buffer geoprocessing tool three different ways: 
    - the ArcGIS Pro GUI
    - ModelBuilder
    - arcpy
- Write an `arcpy.analysis.Buffer` call with a workspace set and a correctly formatted distance string.
- Find a tool's parameter reference inside ArcGIS Pro instead

## The task: buffer every state park by 1 mile

Imagine... Ohio DNR has asked you a simple question: "Which land falls within 1 mile of a state park?" To answer it you need one geoprocessing operation, **Buffer**, using `ohio_state_parks.shp`, the layer of Ohio's state park boundaries.

But... ArcGIS Pro doesn't provide one way to run Buffer. It gives you three. Same tool, same parameters, same result — but three very different experiences implementations

## Implementation 1: The GUI dialog

<img
  src="./images/wk02_gui.png"
  alt="a screenshot of the ArcGIS Pro graphical user interface demonstrating how to run the Buffer tool"
  width="500"
/>

This is the way most GIS software has worked for decades: open a tool, fill in a form, click OK.

1. In the **Catalog** pane, expand **Toolboxes → Analysis Tools → Proximity**, and double-click **Buffer**. (Or search "buffer" in the Geoprocessing pane's search box)
2. **Input Features:** `ohio_state_parks.shp`
3. **Output Feature Class:** `parks_1mi_gui.shp` — ArcGIS Pro will suggest a path in the current workspace; accept it or browse to something sensible you have access to
4. **Distance:** type `1` and choose **Miles** from the unit dropdown
5. Leave everything else at its default for now — we'll come back to some of these options next session
6. Click **Run**

**Predict before you run:** Before you click Run, how many output polygons do you expect `parks_1mi_gui.shp` to have? More than, fewer than, or exactly the same as `ohio_state_parks.shp` has features?

**What was the result?**

## Implementation 2: ModelBuilder

ModelBuilder is ArcGIS Pro's "visual programming canvas." Using it, you connect tools together as boxes and arrows instead of typing code. For a single Buffer call it's overkill, but it's also how repeatable, multi-step workflows are built without writing a line of Python. Further, it previews what "chaining tools together" looks like before we do so in code later this semester.

1. On the **Insert** tab, click **New Model**. A blank canvas opens along with the **ModelBuilder** ribbon.
2. Drag **Buffer** from the Geoprocessing pane's search results onto the canvas. It will appear as an unfilled tool box.
3. Double-click the Buffer box to open its parameter dialog (same parameters as Implementation 1): Input Features = `ohio_state_parks.shp`, Output = `parks_1mi_model.shp`, Distance = `1 Miles`. Click OK.
4. The box will fill in with color once every required parameter is set, and ArcGIS Pro automatically draws the input/output ovals connected to it.
5. Right-click any parameter oval and choose **Model Parameter** to expose it. This will turn your model into a small tool with its own dialog, so someone else could run it without opening ModelBuilder at all.
6. Save the model, then click **Run** on the ModelBuilder ribbon (or right-click the Buffer box → **Run**).


<img
  src="./images/wk02_model.png"
  alt="a screenshot of the ModelBuilder interface in ArcGIS Pro, showing how to build a simple model with the Buffer tool"
  width="500"
/>

**Predict before you run:** Imagine that you expose `Distance` as a model parameter and hand a copy of your project to a classmate. What do you think changes about how *they* interact with your model, compared to how you just built it?

**Expected result from Implementation 2:** `parks_1mi_model.shp` appears alongside `parks_1mi_gui.shp`. You'll have the same shape, same count (~67), same polygons. This makes sense, since it's the same tool with the same inputs. But the difference is what you're left with after running the tool. In this case, you have a saved, reusable model in your toolbox that anyone (with the project) can open and re-run, complete with its own dialog if you exposed parameters.

### Your turn:

- Pick a different dataset: `ohio_counties.shp` (88 polygons) or `portage_adj_munis.shp` (186 polygons)

- And choose a different distance, anywhere from `"500 Meters"` to `"3 Miles"`. 

- Repeat Implementation 1's GUI steps on it, then repeat Implementation 2's ModelBuilder steps on that same layer and distance. Before you click Run either time, predict the output feature count.

Or, from the variety folder: buffer a campus building or path you walk daily (`variety/kent_campus_buildings.shp`, 153 polygons, or `variety/kent_paths.shp`, 1,625 lines) by a small distance like `"25 Feet"` 

## Implementation 3: arcpy

arcpy is ArcGIS Pro's Python package for geoprocessing. Every tool you can implement in the GUI has an arcpy equivalent

Open a new Python notebook in Pro (**Insert → New Notebook**) for this part.

First, arcpy needs to know where to find your data and where to put its outputs — that's the **workspace**

## But first, workspaces... Wait, what's a workspace?

A **workspace** is the folder (or geodatabase) arcpy treats as "home." Once you set one, every tool call that doesn't explicitly state a full path gets resolved against it — `"ohio_state_parks.shp"` means "the file called that, inside my workspace." 

You'll type arcpy tool calls constantly, and full paths are long. Set a workspace once at the top of your notebook, and every filename after that stays short and readable.

## The backslash problem

Windows shows you paths with backslashes: `C:\Users\username\desktop\week02`. Typing that directly into a Python string is a trap.

**Predict before you run:** The cell below prints a Windows-style path exactly as you'd copy it from File Explorer, backslashes and all. Guess what actually prints — is it the same text you typed?

In [ ]:
windows_style_path = "C:\Users\username\desktop\activity02"
print(windows_style_path)

### What happened?

Python treats a backslash inside a string as the start of an **escape sequence**, a special two-character code.

 `\a` happens to be a real one (it means "bell," an old terminal-alert character), so it silently disappears from the printed text instead of raising an error. 

(`\u` inside a string lets you insert characters using their Unicode hex code points. This bug is inconsistent and hard to catch by eye. Different letters after a backslash behave differently.)

## Three fixes

1. **Forward slashes** — `"C:/Users/username/desktop/activity02"`. Windows accepts forward slashes in paths just fine, and Python never treats `/` as an escape character. **This is the default for this course** — use it unless you have a specific reason not to.
2. **Raw strings** — `r"C:\Users\username\desktop\activity02"`. The `r`  before the opening quote tells Python "don't process escape sequences in this string." You'll see this a lot in other people's code; it's fine to use, just don't forget the `r`.
3. **Doubled backslashes** — `"C:\\Users\\username\\desktop\\activity02"`. `\\` is the escape sequence for a single literal backslash. It works, but it's the least readable of the three — easy to miscount backslashes in a long path.

In [ ]:
forward_slash_path = "C:/Users/username/desktop/actvity02"
raw_string_path = r"C:\Users\username\desktop\activity02"
doubled_backslash_path = "C:\\Users\\username\\desktop\\activity02"

print(forward_slash_path)
print(raw_string_path)
print(doubled_backslash_path)

## Setting the workspace and allowing overwrite

**Predict before you run:** You're about to set `arcpy.env.workspace` to your working folder and `arcpy.env.overwriteOutput` to `True`. If you re-run a Buffer call a second time with the same output name, what do you think happens with `overwriteOutput` set to `True`, versus left at its default (`False`)?

***Edit the below cell to point to a directory somewhere you have access to (e.g., your desktop, external drive, wherever)***

In [ ]:
import arcpy

arcpy.env.workspace = "C:/Users/YOURUSERNAMEHERE/Desktop/Activity02"
arcpy.env.overwriteOutput = True

**Expected result:** 

nothing prints. But now, if you re-run any tool call in this notebook that writes to a file that already exists, arcpy replaces it *quietly* instead of raising an `ExecuteError` telling you the output already exists. 

That can be convenient while you're iterating on a notebook (you'll re-run cells constantly), but it's also a way to silently overwrite a result you meant to keep. Turn it off (or rename your outputs) once you're past the exploring stage.

**NOTE** setting a workspace doesn't produce output, it just changes what arcpy assumes about relative filenames from here on. Every geoprocessing call in this notebook now treats your workspace as "home".

### Let's actually run the tool

Now the actual tool call. ArcGIS Pro's modern arcpy syntax groups tools by toolbox. For example, `Buffer` in the analysis toolbox, so it's `arcpy.analysis.Buffer()`. (You'll sometimes see the older `Buffer_analysis(...)` form in tutorials online...it still works, but this course uses the modern dotted form)

**Predict before you run:** This line does the exact same operation as Implementation 1 and Implementation 2 — buffer `ohio_state_parks.shp` by 1 mile. Guess what, if anything, prints when it finishes.

In [ ]:
arcpy.analysis.Buffer("ohio_state_parks.shp", "parks_1mi.shp", "1 Mile")

**Expected result:** `parks_1mi.shp` appears in your workspace folder, and it should be the same shape as `parks_1mi_gui.shp` and `parks_1mi_model.shp`. 

The cell itself may print a `<Result 'parks_1mi.shp'>` line or nothing at all, depending on your ArcGIS Pro version. Check the Catalog pane or list the folder to confirm the file was written to disk.

Discussion point:
- What DIDN'T change between the different methods?
- What DID change?

**Predict before you run:** Every layer you've buffered today has been polygons. 

`streams_portage.geojson` is different: it's line data, 1,989 stream and river segments confined to Portage County, and it's a GeoJSON file rather than a shapefile. It adds to your map and works with `arcpy.analysis.Buffer` the same way a shapefile does

You're about to buffer it by 100 feet, same tool, same `dissolve_option="NONE"` default as `parks_1mi.shp`. 

**Guess:** will the output have more, fewer, or the same number of features as the 1,989 input lines? What shape do you expect a buffered *line* to make, compared to the ring you got from a buffered polygon?

In [ ]:
arcpy.analysis.Buffer("streams_portage.geojson", "streams_100ft.shp", "100 Feet")

**Your turn:** Using the workspace already set above, buffer `ohio_counties.shp` by `"250 Meters"` into an output name of your choosing. Before running, predict the feature count by comparing to the parks and streams examples above, then check it against the output's attribute table.

Or, from the variety folder: buffer `kent_campus_buildings.shp` or `kent_paths.shp` by the same `"250 Meters"`

In [ ]:
# Try it here:



## Running a tool with more parameters: Buffer's dissolve option

So far Buffer has only used its first three parameters (input, output, distance). Real tool calls often set more. Buffer, for example, takes a `dissolve_option` parameter. If you retain the default `"NONE"`, you get one output polygon per input feature (which is what happened earlier); set it to `"ALL"` and every buffer gets merged into one single feature (whether or not any of them touch)

In [ ]:
arcpy.analysis.Buffer(
    "ohio_state_parks.shp",
    "parks_1mi_dissolved.shp",
    "1 Mile",
    dissolve_option="ALL",
)

**Review:** How is it different than before?

## Benefits of different implementation modes:

| Mode | Best for | Cost |
|---|---|---|
| GUI dialog | A true one-off - you'll never run this exact operation again | Fast to start,  leaves no record of what you did; you'd have to remember the parameters to repeat it |
| ModelBuilder | A workflow you'll rerun yourself, or hand to a non-programmer colleague | More setup than the GUI, but the model is a reusable, inspectable artifact |
| arcpy | Anything you'll repeat, batch, schedule, or need to explain precisely (in code, not memory) | Steepest learning curve, but cheapest to rerun, and easiest to version and share |

There's no universally "best" mode

## Checking existence before you run

One of the most common arcpy failures is trying to read a file that isn't where you think it is — a typo in a filename, a workspace pointed at the wrong folder, a file that hasn't been created yet. `arcpy.Exists()` checks before you find out the hard way.


**Predict before you run:** The cell below checks two names — one that's real (`ohio_counties.shp`) and one that isn't (`ohio_countys.shp`, missing the "e"). Guess what each check returns.

In [ ]:
real_name = "ohio_counties.shp"
typo_name = "ohio_countys.shp"

print(f"{real_name} exists: {arcpy.Exists(real_name)}")
print(f"{typo_name} exists: {arcpy.Exists(typo_name)}")

**What happened???**

**Your turn:** With a BUDDY:

Write an `if arcpy.Exists(...):` check around a `Buffer` call of your choosing, so the tool only runs when its input is actually present. Test it two ways: once with a real layer name (it should run), once with a typo'd name (it should skip silently, no error). You won't need `try`/`except` for this until later in the class — a plain `if` is enough for now.

In [ ]:
# Try it here:



## Building a habit of using reference documentation

Every arcpy tool has an ArcGIS Pro help page describing its exact parameters, valid values, and defaults. You'll want it, because there's no reasonable way to memorize dozens of tools' full parameter lists.

You'll sometimes see old tutorials call `arcpy.Usage("Buffer_analysis")` from the Python window to print a tool's syntax. **Don't build that habit.** It's tied to the legacy tool-name syntax this course doesn't use, and it gives you a bare syntax string with none of the explanation, examples, or valid-value lists you actually need.

Instead: in the Geoprocessing pane, every tool dialog has a small **?** or "Learn more about [Tool]" link at the bottom — click it, and ArcGIS Pro opens that tool's full reference page in your browser. That's the same page you'd find by searching "[tool name] arcgis pro" online, and it's worth bookmarking for the handful of tools you'll use most (Buffer, Clip, Select, Dissolve). When you're unsure what a parameter does or what values it accepts, that page is the answer.

## For next class

- Readings on Canvas
- Sketch 1 due Thursday
- Lab 01 due next week
- We WILL NOT MEET ON THURSDAY - START LAB 01!